# Model Development - UCI Bank Marketing Dataset

**Project:** CAP182 Capstone - STADIOEquities First-Deposit Activation
**Part:** B - Model Development 
**Input:** `data/processed/bank_marketing_features.csv` (output of the feature-engineering
stage)

---

This notebook develops and validates a **Logistic Regression** classification model using the feature engineered dataset. The model is trained and evaluated on the prepared data using a consistent validation framework. No preprocessing or feature engineering is repeated in this notebook; the validated, feature-engineered dataset is loaded and used as provided. 

## 1. Load the Feature-Engineered Dataset

The preprocessing and feature engineering stages have prepared the dataset for model development. The resulting dataset contains numerical predictor variables and the binary target variable `y_binary`.

This notebook develops a Logistic Regression classification model using the saved feature engineered dataset.

The UCI Bank Marketing dataset is used as a structural proxy for the STADIOEquities activation problem. The target variable represents whether a client subscribed to a term deposit and is therefore used as a proxy for the project's financial conversion outcome. The model results are not used to make claims about actual STADIOEquities customers.

The feature engineered dataset is loaded from the processed-data folder so that this notebook can be run independently from the earlier preprocessing and feature engineering notebooks.

In [1]:
import pandas as pd
import numpy as np

# Load the validated feature-engineered dataset
FEATURES_PATH = "data/processed/bank_marketing_features.csv"

df = pd.read_csv(FEATURES_PATH)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully.
Rows: 45211
Columns: 51


**1.1 Verify the Target and Predictor Variables**

The target variable for this classification problem is `y_binary`, where:

- `0` represents the non-conversion outcome
- `1` represents the conversion outcome

All remaining variables are treated as predictor variables.

Before splitting the data, the target distribution and predictor structure are checked to confirm that the feature-engineered dataset is suitable for model development.

In [3]:
# Separating the target from the predictor variables
X = df.drop(columns=["y_binary"])
y = df["y_binary"]

# Checking the target distribution
print("Target distribution:")
print(y.value_counts())

# Checking the target proportions
print("\nTarget proportions:")
print(y.value_counts(normalize=True).round(4))

# Confirming the predictor structure
print("\nPredictor dimensions:")
print("Rows:", X.shape[0])
print("Predictors:", X.shape[1])

# Confirming that all predictors are numerical
print("\nNon-numerical predictors:")
print(X.select_dtypes(exclude=["number"]).columns.tolist())

Target distribution:
y_binary
0    39922
1     5289
Name: count, dtype: int64

Target proportions:
y_binary
0    0.883
1    0.117
Name: proportion, dtype: float64

Predictor dimensions:
Rows: 45211
Predictors: 50

Non-numerical predictors:
[]


## 2. Create the Train/Test Split

The dataset is divided into a training set and a held-out test set before model development.

An 80/20 split is used, with 80% of the observations used for model development and 20% reserved for final evaluation.

Stratification is applied using the target variable so that the proportion of conversion and non-conversion observations remains approximately consistent in both datasets. This is important because the target is imbalanced, with approximately 88.3% of observations belonging to class 0 and 11.7% belonging to class 1.

The test set is kept separate from model development and will only be used for the final evaluation of the Logistic Regression model.

In [4]:
from sklearn.model_selection import train_test_split

# Create an 80/20 stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display the resulting dataset sizes
print("Training set:")
print("Rows:", X_train.shape[0])

print("\nTest set:")
print("Rows:", X_test.shape[0])

# Check that the target proportions are preserved
print("\nTraining target proportions:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTest target proportions:")
print(y_test.value_counts(normalize=True).round(4))

Training set:
Rows: 36168

Test set:
Rows: 9043

Training target proportions:
y_binary
0    0.883
1    0.117
Name: proportion, dtype: float64

Test target proportions:
y_binary
0    0.883
1    0.117
Name: proportion, dtype: float64


## 3. Build the Logistic Regression Model

Logistic Regression is used as the first classification model because the target variable is binary.

The model provides an interpretable baseline for identifying differences between the two outcome classes. It estimates the probability of an observation belonging to the positive class and uses the predictor variables to distinguish between the two outcomes.

The predictor variables are placed in a `Pipeline` with `StandardScaler` followed by Logistic Regression. Scaling is fitted only on the relevant training data, which prevents information from the test set from influencing model development.

A maximum iteration value is specified to provide sufficient iterations for the optimisation process to converge.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Scaling predictors and fit the Logistic Regression model
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Fitting the model using the training data only
logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


## 4. Perform 5-Fold Stratified Cross Validation

Cross validation is used to assess the consistency of the Logistic Regression model during model development.

A 5-fold stratified cross-validation procedure is applied to the training data. Stratification maintains the proportion of the two target classes within each fold.

The held out test set is not used during cross validation. It remains reserved for the final evaluation of the trained model.

Accuracy, precision, recall, F1-score and ROC-AUC are evaluated because the target variable is imbalanced and accuracy alone may provide a misleading assessment of model performance.

In [6]:
from sklearn.model_selection import StratifiedKFold, cross_validate

# Creating a reproducible 5-fold stratified cross validation scheme
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Define the evaluation metrics
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# Evaluating the Logistic Regression pipeline across the training folds
cv_results = cross_validate(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

# Displaying the mean cross validation performance
print("Mean 5-Fold Cross-Validation Results:")

for metric in scoring:
    train_score = cv_results[f"train_{metric}"].mean()
    validation_score = cv_results[f"test_{metric}"].mean()

    print(
        f"{metric.upper()}: "
        f"Training = {train_score:.4f}, "
        f"Validation = {validation_score:.4f}"
    )

Mean 5-Fold Cross-Validation Results:
ACCURACY: Training = 0.8922, Validation = 0.8917
PRECISION: Training = 0.6357, Validation = 0.6271
RECALL: Training = 0.1846, Validation = 0.1829
F1: Training = 0.2861, Validation = 0.2830
ROC_AUC: Training = 0.7747, Validation = 0.7698


The cross validation results provide an initial assessment of Logistic Regression performance during model development. Training and validation scores are relatively similar, while the relatively low recall and F1-score indicate that the model identifies only a limited proportion of the positive class at the default classification threshold. Final performance will be assessed using the held-out test set.

## 5. Evaluate the Model on the Held-Out Test Set

After cross validation, the Logistic Regression pipeline is evaluated using the held-out test set.

The test set was not used during model fitting or cross validation and therefore provides an independent assessment of how the trained model performs on unseen observations.

The evaluation includes accuracy, precision, recall, F1-score and ROC-AUC. A confusion matrix and classification report are also produced to provide a more detailed view of the model's classification performance.

Because the target is imbalanced, particular attention is given to precision, recall and F1-score rather than accuracy alone.

In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Generating predictions for the unseen test set
y_test_pred = logistic_model.predict(X_test)
y_test_prob = logistic_model.predict_proba(X_test)[:, 1]

# Calculating evaluation metrics
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_roc_auc = roc_auc_score(y_test, y_test_prob)

print("Logistic Regression Test-Set Results:")
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-score:  {test_f1:.4f}")
print(f"ROC-AUC:   {test_roc_auc:.4f}")

# Displaying the confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

# Displaying the full classification report
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

Logistic Regression Test-Set Results:
Accuracy:  0.8934
Precision: 0.6556
Recall:    0.1871
F1-score:  0.2912
ROC-AUC:   0.7789

Confusion Matrix:
[[7881  104]
 [ 860  198]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.99      0.94      7985
           1       0.66      0.19      0.29      1058

    accuracy                           0.89      9043
   macro avg       0.78      0.59      0.62      9043
weighted avg       0.87      0.89      0.87      9043



The Logistic Regression model achieved an accuracy of 89.34% and a ROC-AUC of 0.7789 on the held-out test set. However, accuracy should be interpreted cautiously because the target variable is imbalanced.

The model achieved a precision of 65.56% for class 1, meaning that when the model predicted the positive outcome, approximately 66% of those predictions were correct. Recall for class 1 was 18.71%, indicating that the model identified a relatively small proportion of the actual positive cases.

The confusion matrix shows that the model correctly classified 7,881 of the 7,985 class 0 observations, while correctly identifying 198 of the 1,058 class 1 observations. Therefore, although the model demonstrates some ability to distinguish between the two classes, its default 0.5 classification threshold results in many positive cases being classified as class 0.

The similarity between the cross validation and held-out test results provides an initial indication that the model performs consistently on unseen data. Further interpretation and comparison with the Gradient Boosting model will be conducted during the Evaluation stage.

## 6. Interpret Logistic Regression Coefficients

Logistic Regression provides coefficients that indicate the direction and strength of association between each predictor and the model's predicted outcome.

Positive coefficients indicate that higher values of a predictor are associated with higher log-odds of the positive class, while negative coefficients indicate lower log-odds, holding the other predictors in the model constant.

The coefficients are examined to identify the predictors with the strongest positive and negative associations with the modelled outcome.

These associations should not be interpreted as evidence of causation. They describe relationships used by the predictive model on the proxy dataset.

In [9]:
# Extracting the fitted Logistic Regression coefficients
coefficients = logistic_model.named_steps["logistic_regression"].coef_[0]

# Creating a table linking each predictor to its coefficient
coefficient_table = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": coefficients
})

# Calculating odds ratios from the coefficients
coefficient_table["odds_ratio"] = np.exp(
    coefficient_table["coefficient"]
)

# Sorting by coefficient to identify the strongest associations
coefficient_table = coefficient_table.sort_values(
    by="coefficient"
)

print("Strongest negative associations:")
print(coefficient_table.head(10).to_string(index=False))

print("\nStrongest positive associations:")
print(coefficient_table.tail(10).sort_values(
    by="coefficient",
    ascending=False
).to_string(index=False))

Strongest negative associations:
        feature  coefficient  odds_ratio
contact_unknown    -0.582051    0.558751
      month_aug    -0.256037    0.774114
    housing_yes    -0.241091    0.785770
      month_nov    -0.229411    0.795001
      month_jul    -0.226701    0.797159
      month_may    -0.210356    0.810296
       campaign    -0.209554    0.810946
      month_jan    -0.190044    0.826923
 age_band_50-59    -0.189392    0.827462
 age_band_40-49    -0.162295    0.850190

Strongest positive associations:
               feature  coefficient  odds_ratio
      poutcome_success     0.395431    1.485024
      poutcome_unknown     0.233259    1.262708
balance_band_1000-4999     0.199589    1.220901
  previously_contacted     0.189043    1.208093
    education_tertiary     0.131286    1.140294
    balance_band_0-999     0.121468    1.129154
             month_mar     0.103726    1.109296
balance_band_5000-9999     0.091917    1.096274
   education_secondary     0.074362    1.077197
  

**6.1 Interpretation of the Strongest Coefficients**

The Logistic Regression coefficients indicate the direction of association between the predictors and the modelled positive outcome, while the odds ratios provide a more interpretable measure of the change in predicted odds.

Among the strongest negative associations, `contact_unknown` has the largest negative coefficient, with an odds ratio of approximately 0.56. This indicates that, relative to the reference category used during encoding, this feature is associated with lower modelled odds of the positive outcome. Other relatively strong negative associations include `housing_yes`, `campaign`, and several month and age-band indicators.

Among the strongest positive associations, `poutcome_success` has the largest positive coefficient, with an odds ratio of approximately 1.49. `poutcome_unknown`, `balance_band_1000-4999`, and `previously_contacted` also have positive coefficients, indicating higher modelled odds of the positive outcome relative to their respective reference categories.

These results identify variables that the Logistic Regression model uses as predictive signals in the proxy dataset. They should not be interpreted as evidence that these variables cause conversion. The categorical coefficients are also interpreted relative to their reference categories created during one-hot encoding.

The identified associations provide a basis for further investigation and comparison with the Gradient Boosting model during the Evaluation stage.

## 7. Logistic Regression Model Development Summary

The Logistic Regression model was developed using the feature-engineered UCI Bank Marketing dataset as a structural proxy for the STADIOEquities activation problem.

An 80/20 stratified train/test split was used, with 36,168 observations allocated to training and 9,043 observations reserved for final testing. A Logistic Regression pipeline combining `StandardScaler` and `LogisticRegression` was fitted using the training data.

Five-fold stratified cross validation was performed on the training data. The validation results showed an average accuracy of 89.17%, precision of 62.71%, recall of 18.29%, F1-score of 28.30% and ROC-AUC of 76.98%.

On the held-out test set, the model achieved an accuracy of 89.34%, precision of 65.56%, recall of 18.71%, F1-score of 29.12% and ROC-AUC of 77.89%. The similarity between the cross validation and test results provides an initial indication of consistent model performance on unseen data.

The model coefficients also identified several variables associated with the predicted outcome. These associations provide potentially useful predictive signals, but they do not establish causal relationships.

The Logistic Regression results will be carried forward to the Evaluation stage, where they can be considered alongside the results of the Gradient Boosting model. Final conclusions about model performance and suitability will be made only after both models have been evaluated and compared.

In [10]:
# Store the main Logistic Regression results for reference
logistic_summary = pd.DataFrame({
    "metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    "cross_validation": [
        cv_results["test_accuracy"].mean(),
        cv_results["test_precision"].mean(),
        cv_results["test_recall"].mean(),
        cv_results["test_f1"].mean(),
        cv_results["test_roc_auc"].mean()
    ],
    "test_set": [
        test_accuracy,
        test_precision,
        test_recall,
        test_f1,
        test_roc_auc
    ]
})

logistic_summary

,metric,cross_validation,test_set
0,Accuracy,0.891672,0.893398
1,Precision,0.627091,0.655629
2,Recall,0.182939,0.187146
3,F1-score,0.283012,0.291176
4,ROC-AUC,0.769762,0.778949
